# DBSCAN con el dataset NYPD Arrest Data (Year to Date)

Este notebook aplica **DBSCAN** a un caso real: los arrestos registrados por el NYPD.


In [ ]:


import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

## 1. Carga del dataset

In [ ]:
ruta = 'NYPD_Arrest_Data__Year_to_Date_.csv'
df = pd.read_csv(ruta)

print('Filas y columnas:', df.shape)
df.head()

## 2. Exploración inicial

In [ ]:
print(df.columns.tolist())
print('Valores nulos por columna:')
print(df[['Latitude', 'Longitude', 'AGE_GROUP', 'OFNS_DESC']].isna().sum())

## 3. Selección de variables


In [ ]:
datos = df[['Latitude', 'Longitude']].copy()
datos = datos.dropna()

print('Registros con coordenadas válidas:', datos.shape[0])
datos.head()

## 4. Muestra para trabajar

El dataset completo es grande. Para que el notebook vaya fluido, tomamos una muestra aleatoria reproducible.

In [ ]:
muestra = datos.sample(n=15000, random_state=42)
print('Tamaño de la muestra:', muestra.shape)
muestra.head()

## 5. Visualización inicial de los puntos

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(muestra['Longitude'], muestra['Latitude'], s=4, alpha=0.4)
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Arrestos del NYPD (muestra)')
plt.show()

## 6. Escalado de variables

DBSCAN trabaja con distancias.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(muestra)

print('Primeras filas escaladas:')
print(X_scaled[:5])

## 7. Estimación orientativa de `eps`

Una forma habitual de orientar el valor de `eps` es observar la distancia al **k-ésimo vecino más cercano**.

In [ ]:
min_samples = 10
vecinos = NearestNeighbors(n_neighbors=min_samples)
vecinos.fit(X_scaled)

distancias, indices = vecinos.kneighbors(X_scaled)

distancias_ordenadas = np.sort(distancias[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(distancias_ordenadas)
plt.xlabel('Puntos ordenados')
plt.ylabel(f'Distancia al vecino {min_samples}')
plt.title('Gráfico k-distance para orientar eps')
plt.show()

## 8. Aplicación de DBSCAN

Tras observar el gráfico, elegimos un valor inicial de `eps`.

In [ ]:
modelo = DBSCAN(eps=0.08, min_samples=10)
clusters = modelo.fit_predict(X_scaled)

muestra['cluster'] = clusters
muestra.head()

In [ ]:
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
ruido = (clusters == -1).sum()

print('Número de clusters encontrados:', n_clusters)
print('Puntos clasificados como ruido:', ruido)
print('Tamaño de cada etiqueta:')
print(pd.Series(clusters).value_counts().sort_index())

## 9. Visualización de los clusters

In [ ]:
plt.figure(figsize=(9, 7))
plt.scatter(
    muestra['Longitude'],
    muestra['Latitude'],
    c=muestra['cluster'],
    s=5,
    alpha=0.6
)
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Clusters encontrados por DBSCAN')
plt.show()

## 10. Resumen estadístico por cluster

In [ ]:
resumen = (
    muestra[muestra['cluster'] != -1]
    .groupby('cluster')[['Latitude', 'Longitude']]
    .agg(['mean', 'count'])
)
resumen